# SAP Incident Knowledge Assistant

## RAG over structured Excel data, with Google Gemini

Support engineers search a historical incident spreadsheet by hand. It is slow when
the sheet has thousands of rows, when nobody remembers the incident ID, and when a
past incident described the same problem in different words.

This notebook builds a RAG application over that spreadsheet. Ask a question in
plain English, get an answer grounded **only** in the incident history, with the
incident ID and Excel row number cited for every claim.

### What you will build

| Task | Step |
| --- | --- |
| 1–2 | Load and clean the Excel data with Pandas |
| 3 | Turn one Excel row into one LangChain `Document` |
| 4 | Row-aware chunking that keeps records intact |
| 5–6 | Gemini embeddings indexed in FAISS |
| 7–8 | Semantic retrieval + metadata filtering |
| 9–10 | A grounded prompt and the final RAG function |
| 11 | Route calculations to Pandas instead of RAG |

## Architecture

```text
sap_incidents.xlsx  (one incident per row)
        |
        |  Task 1-2   Pandas: load, stamp Excel row number, clean
        v
  Clean dataframe ------------------------------+
        |                                       |
        |  Task 3     one row -> one Document   |
        v             (labelled text + metadata)|
  LangChain Documents                           |
        |                                       |
        |  Task 4     split only if oversized   |
        v                                       |
     Chunks                                     |
        |                                       |
        |  Task 5-6   Gemini embeddings         |
        v                                       |
   FAISS vector store                           |
        |                                       |
        +---------------+-----------------------+
                        |
                  User question
                        |
                        v
              Question classification            (Task 11)
                        |
        +---------------+----------------+
        |                                |
  Semantic question               Analytical question
        |                                |
  Tasks 7-8                       Pandas aggregates
  vector search                   over ALL rows
  + metadata filter                      |
        |                                |
        v                                v
  Retrieved records                 Exact statistics
        |                                |
        +---------------+----------------+
                        |
                  Task 9 grounded prompt
                        v
                  Google Gemini
                        |
                        v
        Answer + incident IDs + Excel row citations
```

## Setup

Install the dependencies, then supply a Gemini API key. Get one free at
[aistudio.google.com/apikey](https://aistudio.google.com/apikey).

In Colab, store it as a secret named `GOOGLE_API_KEY` (the key icon in the left
sidebar) and the next cell will pick it up. Otherwise it will prompt you.

In [ ]:
%pip install -q langchain langchain-core langchain-community langchain-google-genai \
    langchain-text-splitters faiss-cpu pandas openpyxl pydantic python-dotenv

In [ ]:
import os

def _load_api_key() -> None:
    """Read the key from Colab secrets, the environment, or a prompt."""
    if os.getenv("GOOGLE_API_KEY"):
        print("Using GOOGLE_API_KEY from the environment.")
        return
    try:
        from google.colab import userdata

        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        print("Using GOOGLE_API_KEY from Colab secrets.")
        return
    except Exception:
        pass

    from getpass import getpass

    os.environ["GOOGLE_API_KEY"] = getpass("Enter your GOOGLE_API_KEY: ")
    print("Key set for this session.")

_load_api_key()

## The dataset

`sap_incidents.xlsx` holds 20 resolved incidents across SAP MM, SD, HANA, BTP and
SuccessFactors.

The first eight rows are the sample data from the problem statement, so **INC-1006
lands on Excel row 7** exactly as the expected output requires. Some later rows are
deliberately messy — an empty row, padded whitespace, a lower-case priority, a
number stored as text, a missing root cause — so that the cleaning step has real
work to do.

In [ ]:
from __future__ import annotations

import pandas as pd

COLUMNS = [
    "incident_id",
    "incident_date",
    "sap_module",
    "category",
    "priority",
    "issue_summary",
    "issue_description",
    "root_cause",
    "resolution",
    "owner_team",
    "resolution_time_hours",
    "status",
]

# --------------------------------------------------------------------------- #
# Rows 2-9: the eight incidents from the problem statement, verbatim.
# --------------------------------------------------------------------------- #
SAMPLE_ROWS = [
    {
        "incident_id": "INC-1001",
        "incident_date": "2024-01-12",
        "sap_module": "SAP MM",
        "category": "Invoice Management",
        "priority": "P2",
        "issue_summary": "Supplier invoice blocked because of price variance",
        "issue_description": (
            "The supplier invoice posted through MIRO was blocked for payment. "
            "The invoice value did not match the purchase-order value and the "
            "system raised a price-variance block, so the payment run skipped it."
        ),
        "root_cause": "Incorrect purchase-order condition record",
        "resolution": "Corrected condition record and reprocessed invoice",
        "owner_team": "Procure-to-Pay Support",
        "resolution_time_hours": 3.5,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1002",
        "incident_date": "2024-01-19",
        "sap_module": "SAP HANA",
        "category": "Database Availability",
        "priority": "P1",
        "issue_summary": "HANA database became unavailable",
        "issue_description": (
            "The production HANA database stopped accepting connections during "
            "the month-end close. The index server terminated and users could "
            "not log on to any connected application."
        ),
        "root_cause": "Severe memory exhaustion",
        "resolution": (
            "Released memory, restarted service and adjusted allocation limits"
        ),
        "owner_team": "Database Platform Team",
        "resolution_time_hours": 5.0,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1003",
        "incident_date": "2024-02-02",
        "sap_module": "SAP SD",
        "category": "Pricing",
        "priority": "P3",
        "issue_summary": "Incorrect discount in sales order",
        "issue_description": (
            "Sales orders created for a customer group applied a larger discount "
            "than the agreed contract value, which understated the net order value."
        ),
        "root_cause": "Incorrect pricing procedure sequence",
        "resolution": "Corrected condition sequence and repriced order",
        "owner_team": "Order-to-Cash Support",
        "resolution_time_hours": 1.2,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1004",
        "incident_date": "2024-02-14",
        "sap_module": "SAP BTP",
        "category": "Connectivity",
        "priority": "P1",
        "issue_summary": "Application could not connect to backend system",
        "issue_description": (
            "A Cloud Foundry application on SAP BTP failed every call to the "
            "on-premise backend with an authentication error. The destination "
            "test connection also failed from the cockpit."
        ),
        "root_cause": "Expired destination credentials",
        "resolution": "Renewed credentials and corrected destination properties",
        "owner_team": "BTP Platform Team",
        "resolution_time_hours": 7.5,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1005",
        "incident_date": "2024-02-21",
        "sap_module": "SAP SuccessFactors",
        "category": "Integration",
        "priority": "P2",
        "issue_summary": "Employee replication was delayed",
        "issue_description": (
            "Employee master data replication from SuccessFactors to the ERP "
            "system stopped for two days. New joiners were missing in payroll."
        ),
        "root_cause": "Mapping error in integration flow",
        "resolution": "Corrected mapping and reran integration job",
        "owner_team": "HR Integration Team",
        "resolution_time_hours": 4.0,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1006",
        "incident_date": "2024-03-05",
        "sap_module": "SAP HANA",
        "category": "Performance",
        "priority": "P1",
        "issue_summary": "Critical transactions became slow",
        "issue_description": (
            "Users experienced severe performance degradation. Business-critical "
            "transactions that normally finish in seconds took several minutes, "
            "and the delta merge queue kept growing."
        ),
        "root_cause": "Long-running savepoint and storage pressure",
        "resolution": "Corrected storage pressure and optimized workload",
        "owner_team": "Database Platform Team",
        "resolution_time_hours": 9.0,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1007",
        "incident_date": "2024-03-11",
        "sap_module": "SAP MM",
        "category": "Purchase Order",
        "priority": "P3",
        "issue_summary": "Purchase order release was not triggered",
        "issue_description": (
            "Purchase orders above the approval threshold were saved without "
            "entering the release workflow, so buyers could order without approval."
        ),
        "root_cause": "Incorrect release strategy configuration",
        "resolution": "Corrected release strategy and regenerated classification",
        "owner_team": "Procure-to-Pay Support",
        "resolution_time_hours": 2.5,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1008",
        "incident_date": "2024-03-18",
        "sap_module": "SAP BTP",
        "category": "Authorization",
        "priority": "P2",
        "issue_summary": "User could not access deployed application",
        "issue_description": (
            "A business user opened the application URL and received a 403 "
            "Forbidden page. Other users in the same department had no problem."
        ),
        "root_cause": "Missing role collection assignment",
        "resolution": "Assigned required role collection",
        "owner_team": "BTP Platform Team",
        "resolution_time_hours": 1.8,
        "status": "Closed",
    },
]

# --------------------------------------------------------------------------- #
# Rows 10+: extra incidents, including deliberately messy ones.
# --------------------------------------------------------------------------- #
EXTRA_ROWS = [
    # A completely empty row -- Task 2 must drop it.
    {c: None for c in COLUMNS},
    {
        "incident_id": "INC-1009",
        "incident_date": "2024-03-26",
        "sap_module": "  sap hana  ",  # inconsistent case + padding
        "category": "Backup",
        "priority": "p2",  # lower case priority
        "issue_summary": "Nightly backup job failed",
        "issue_description": (
            "The scheduled full backup of the HANA production tenant failed with "
            "a write error and no usable backup existed for that day."
        ),
        "root_cause": "Backup target file system was full",
        "resolution": "Freed space on the backup volume and reran the full backup",
        "owner_team": "Database Platform Team",
        "resolution_time_hours": "2.75",  # numeric stored as text
        "status": "Closed",
    },
    {
        "incident_id": "INC-1010",
        "incident_date": "01-04-2024",  # different date format
        "sap_module": "SAP BTP",
        "category": "Connectivity",
        "priority": "P2",
        "issue_summary": "Cloud Connector tunnel dropped intermittently",
        "issue_description": (
            "The Cloud Connector lost its tunnel to the BTP subaccount several "
            "times a day. Applications calling the on-premise system failed "
            "during each outage window."
        ),
        "root_cause": "Firewall idle-timeout closed the tunnel connection",
        "resolution": (
            "Adjusted the firewall idle timeout and enabled keep-alive on the "
            "Cloud Connector"
        ),
        "owner_team": "BTP Platform Team",
        "resolution_time_hours": 6.0,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1011",
        "incident_date": "2024-04-08",
        "sap_module": "SAP SD",
        "category": "Billing",
        "priority": "P2",
        "issue_summary": "Billing documents stuck in the accounting interface",
        "issue_description": (
            "Billing documents were created but not released to accounting, so "
            "revenue was missing from the general ledger."
        ),
        "root_cause": "Missing account determination for a new material group",
        "resolution": "Maintained account determination and released the documents",
        "owner_team": "Order-to-Cash Support",
        "resolution_time_hours": 3.0,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1012",
        "incident_date": "2024-04-15",
        "sap_module": "SAP MM",
        "category": "Inventory",
        "priority": "P1",
        "issue_summary": "Goods receipt postings failed during shift start",
        "issue_description": (
            "Warehouse staff could not post any goods receipt. Every posting "
            "terminated with a number-range error and the receiving dock stopped."
        ),
        "root_cause": "Material document number range was exhausted",
        "resolution": "Extended the number range interval and reposted the documents",
        "owner_team": "Procure-to-Pay Support",
        "resolution_time_hours": 4.5,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1013",
        "incident_date": "2024-04-22",
        "sap_module": "SAP SuccessFactors",
        "category": "Authorization",
        "priority": "P3",
        "issue_summary": "Managers could not see their team in the org chart",
        "issue_description": (
            "Several managers reported an empty org chart although their direct "
            "reports were maintained correctly in employee central."
        ),
        "root_cause": None,  # missing text field -- Task 2 fills a default
        "resolution": "Rebuilt the role-based permission group and refreshed the cache",
        "owner_team": "HR Integration Team",
        "resolution_time_hours": 2.0,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1014",
        "incident_date": "2024-05-03",
        "sap_module": "SAP HANA",
        "category": "Performance",
        "priority": "P2",
        "issue_summary": "Reporting queries slowed down after data load",
        "issue_description": (
            "Analytical queries on the sales tables became progressively slower "
            "after the nightly load finished."
        ),
        "root_cause": "Delta storage was not merged after the bulk load",
        "resolution": "Triggered a delta merge and tuned the auto-merge settings",
        "owner_team": "Database Platform Team",
        "resolution_time_hours": 3.25,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1015",
        "incident_date": "2024-05-10",
        "sap_module": "SAP BTP",
        "category": "Integration",
        "priority": "P1",
        "issue_summary": "Integration Suite iFlow stopped processing messages",
        "issue_description": (
            "Messages piled up in the queue and none reached the receiver "
            "system. The iFlow showed a permanent error status."
        ),
        "root_cause": "Expired client certificate on the receiver channel",
        "resolution": "Deployed a renewed certificate and restarted the iFlow",
        "owner_team": "BTP Platform Team",
        "resolution_time_hours": 8.0,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1016",
        "incident_date": "2024-05-17",
        "sap_module": "SAP SD",
        "category": "Delivery",
        "priority": "P2",
        "issue_summary": "Deliveries could not be picked in the warehouse",
        "issue_description": (
            "Outbound deliveries were created but picking could not start, so "
            "trucks waited at the dock."
        ),
        "root_cause": "Shipping point was not assigned to the new plant",
        "resolution": "Assigned the shipping point and reprocessed the deliveries",
        "owner_team": "Order-to-Cash Support",
        "resolution_time_hours": 2.2,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1017",
        "incident_date": "2024-05-24",
        "sap_module": "SAP HANA",
        "category": "Database Availability",
        "priority": "P3",
        "issue_summary": "Secondary system fell behind in system replication",
        "issue_description": (
            "The replication delay on the secondary site grew beyond the agreed "
            "recovery point objective, putting failover readiness at risk."
        ),
        "root_cause": "Network bandwidth saturation between data centres",
        "resolution": "Rescheduled bulk traffic and re-established replication",
        "owner_team": "Database Platform Team",
        "resolution_time_hours": 5.5,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1018",
        "incident_date": "2024-06-01",
        "sap_module": "SAP MM",
        "category": "Invoice Management",
        "priority": "P2",
        "issue_summary": "Duplicate supplier invoices were posted",
        "issue_description": (
            "The same supplier invoice was posted twice in the same week, which "
            "created a risk of double payment."
        ),
        "root_cause": "Duplicate invoice check was not active for the company code",
        "resolution": "Activated the duplicate check and reversed the extra document",
        "owner_team": "Procure-to-Pay Support",
        "resolution_time_hours": 3.8,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1019",
        "incident_date": "2024-06-09",
        "sap_module": "SAP SuccessFactors",
        "category": "Integration",
        "priority": "P1",
        "issue_summary": "Payroll interface failed before the pay run",
        "issue_description": (
            "The compensation data transfer to payroll aborted, and the pay run "
            "could not start on the planned date."
        ),
        "root_cause": "Unmapped pay component introduced by a configuration change",
        "resolution": "Added the missing mapping and reran the interface",
        "owner_team": "HR Integration Team",
        "resolution_time_hours": 6.5,
        "status": "Closed",
    },
    {
        "incident_id": "INC-1020",
        "incident_date": "2024-06-14",
        "sap_module": "SAP BTP",
        "category": "Authorization",
        "priority": "P3",
        "issue_summary": "Service key rotation broke a scheduled job",
        "issue_description": (
            "A background job calling a BTP service started failing with an "
            "unauthorized error after a routine key rotation."
        ),
        "root_cause": "Job still referenced the old service key",
        "resolution": "Updated the job configuration with the new service key",
        "owner_team": "BTP Platform Team",
        "resolution_time_hours": 1.5,
        "status": "Open",
    },
]


def build_dataframe() -> pd.DataFrame:
    """Return the raw (deliberately imperfect) incident dataframe."""
    return pd.DataFrame(SAMPLE_ROWS + EXTRA_ROWS, columns=COLUMNS)


def main() -> None:
    df = build_dataframe()
    df.to_excel("sap_incidents.xlsx", sheet_name="incidents", index=False)
    print(f"Wrote sap_incidents.xlsx with {len(df)} rows (sheet: incidents).")
    print("INC-1006 is on Excel row", df.index[df["incident_id"] == "INC-1006"][0] + 2)


main()

## Imports and configuration

`GEMINI_CHAT_MODEL` and `GEMINI_EMBED_MODEL` are read from the environment so the
notebook keeps working as new Gemini versions ship.

`MODULE_CANON` matters more than it looks: the raw sheet contains `"  sap hana  "`
as well as `"SAP HANA"`. Left alone, that row becomes invisible to a
`sap_module="SAP HANA"` filter forever.

In [ ]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass
from typing import Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

load_dotenv()

# Model names are overridable via environment variables so the app keeps working
# as new Gemini versions are released.
CHAT_MODEL = os.getenv("GEMINI_CHAT_MODEL", "gemini-2.0-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "models/gemini-embedding-001")

EXCEL_FILE = "sap_incidents.xlsx"
SHEET_NAME = "incidents"

# A record longer than this is split; every row in this dataset stays well under
# it, so in practice one incident remains one chunk.
MAX_CHARS_PER_CHUNK = 1500
CHUNK_OVERLAP = 150

TEXT_COLUMNS = [
    "incident_id",
    "sap_module",
    "category",
    "priority",
    "issue_summary",
    "issue_description",
    "root_cause",
    "resolution",
    "owner_team",
    "status",
]

# Canonical spelling for every module, keyed by its normalised form. Excel data
# arrives as "  sap hana  ", "SAP Hana", "SAP HANA" and all must collapse to one
# value, otherwise a metadata filter on sap_module silently misses rows.
MODULE_CANON = {
    "sap mm": "SAP MM",
    "sap sd": "SAP SD",
    "sap hana": "SAP HANA",
    "sap btp": "SAP BTP",
    "sap successfactors": "SAP SuccessFactors",
}

MISSING_TEXT = "Not documented"

## Task 1 — Load and explore the Excel data

The important line is `df["excel_row"] = df.index + 2`.

Row 1 of the workbook is the header, so the first incident sits on Excel row 2.
The row number is stamped **before any cleaning**, because dropping the empty row
later would shift every subsequent row and silently corrupt every citation.

In [ ]:
def load_incidents(path: str = EXCEL_FILE, sheet: str = SHEET_NAME) -> pd.DataFrame:
    """Read the workbook and stamp each record with its true Excel row number.

    Row 1 holds the header, so the first data row is Excel row 2. This happens
    before cleaning because dropping rows would otherwise shift every citation.
    """
    df = pd.read_excel(path, sheet_name=sheet, engine="openpyxl")
    df["excel_row"] = df.index + 2
    df.attrs["source_name"] = os.path.basename(path)
    df.attrs["sheet_name"] = sheet
    return df


def explore_incidents(df: pd.DataFrame) -> None:
    """Print the Task 1 data-quality checks."""
    print("=" * 70)
    print("TASK 1 -- DATA EXPLORATION")
    print("=" * 70)

    print(f"\nRecords: {len(df)}   Columns: {len(df.columns)}")

    print("\nFirst five rows:")
    print(df.head().to_string())

    print("\nColumn names:")
    print(list(df.columns))

    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nMissing values per column:")
    missing = df.isna().sum()
    print(missing[missing > 0].to_string() if missing.any() else "None")

    print("\nIncidents by SAP module:")
    print(df["sap_module"].value_counts(dropna=False).to_string())

    print("\nIncidents by priority:")
    print(df["priority"].value_counts(dropna=False).to_string())

## Task 2 — Clean and prepare the data

The raw sheet is deliberately messy, and each defect maps to a fix:

| Defect in the sheet | Fix |
| --- | --- |
| A completely empty row | `dropna(how="all")` |
| `"  sap hana  "` | `_canonical_module` → `SAP HANA` |
| `"p2"` | `.str.upper()` → `P2` |
| `"2.75"` stored as text | `pd.to_numeric` |
| A missing root cause | default `"Not documented"` |
| `2024-03-05` mixed with `01-04-2024` | `_parse_dates` |

`_parse_dates` deserves a note. Passing `dayfirst=True` across the whole column
would reinterpret the ISO dates and swap day for month — `2024-03-05` would
silently become 3 May. So ISO is parsed strictly first, and only the leftovers
get `dayfirst`.

In [ ]:
def _clean_text(value) -> str:
    """Strip padding, collapse runs of whitespace, drop control characters."""
    if pd.isna(value):
        return MISSING_TEXT
    text = str(value).replace(" ", " ")
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or MISSING_TEXT


def _canonical_module(value: str) -> str:
    return MODULE_CANON.get(value.strip().lower(), value.strip())


def _parse_dates(series: pd.Series) -> pd.Series:
    """Normalise mixed date formats to YYYY-MM-DD.

    The raw file holds both 2024-03-05 and 01-04-2024. Passing dayfirst=True over
    the whole column would reinterpret the ISO dates and silently swap day and
    month, so ISO is parsed strictly first and only the leftovers get dayfirst.
    """
    iso = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
    remainder = pd.to_datetime(series[iso.isna()], errors="coerce", dayfirst=True)
    parsed = iso.fillna(remainder)
    return parsed.dt.strftime("%Y-%m-%d").fillna("Unknown")


def clean_incidents(df: pd.DataFrame) -> pd.DataFrame:
    """Return a clean dataframe ready to become LangChain documents."""
    print("\n" + "=" * 70)
    print("TASK 2 -- CLEANING")
    print("=" * 70)

    before = len(df)
    data_columns = [c for c in df.columns if c != "excel_row"]

    # Remove rows that carry no data at all.
    df = df.dropna(how="all", subset=data_columns).copy()
    print(f"\nDropped {before - len(df)} completely empty row(s).")

    for column in TEXT_COLUMNS:
        df[column] = df[column].map(_clean_text)

    df["sap_module"] = df["sap_module"].map(_canonical_module)
    df["priority"] = df["priority"].str.upper()
    df["status"] = df["status"].str.title()

    df["incident_date"] = _parse_dates(df["incident_date"])

    # Resolution time arrives as float in most rows and as text in others.
    df["resolution_time_hours"] = pd.to_numeric(
        df["resolution_time_hours"], errors="coerce"
    )

    # An incident with no ID cannot be cited, so it cannot be trusted.
    df = df[df["incident_id"] != MISSING_TEXT]

    df = df.reset_index(drop=True)
    print(f"Clean records: {len(df)}")
    print(f"Modules after normalisation: {sorted(df['sap_module'].unique())}")
    print(f"Priorities after normalisation: {sorted(df['priority'].unique())}")
    return df

## Task 3 — Convert Excel rows into documents

**One row becomes exactly one `Document`.** A row *is* the unit of meaning: the
symptom, the root cause, the fix and the team only make sense together, and
`"Expired destination credentials"` is useless without the symptom it explains.

The field labels stay in the text on purpose. The embedding then carries the
meaning of each value (`Root Cause: ...`) rather than a bare string, and the model
can quote the record back without guessing what a column meant.

The structured fields are *also* kept as metadata, which is what makes filtering
and citation possible later.

In [ ]:
def row_to_text(row: pd.Series) -> str:
    """Render one incident as the labelled text block the LLM will read.

    The field labels stay in the text on purpose: the embedding then carries the
    meaning of each value ("Root Cause: ...") rather than a bare string, and the
    model can quote the record back without guessing what a column means.
    """
    hours = row["resolution_time_hours"]
    hours_text = "Unknown" if pd.isna(hours) else f"{hours:g} hours"
    return (
        f"Incident ID: {row['incident_id']}\n"
        f"Incident Date: {row['incident_date']}\n"
        f"SAP Module: {row['sap_module']}\n"
        f"Priority: {row['priority']}\n"
        f"Category: {row['category']}\n"
        f"Issue Summary: {row['issue_summary']}\n"
        f"Issue Description: {row['issue_description']}\n"
        f"Root Cause: {row['root_cause']}\n"
        f"Resolution: {row['resolution']}\n"
        f"Owner Team: {row['owner_team']}\n"
        f"Resolution Time: {hours_text}\n"
        f"Status: {row['status']}"
    )


def rows_to_documents(df: pd.DataFrame) -> List[Document]:
    """Convert every cleaned Excel row into exactly one LangChain Document."""
    source_name = df.attrs.get("source_name", EXCEL_FILE)
    sheet_name = df.attrs.get("sheet_name", SHEET_NAME)

    documents = []
    for _, row in df.iterrows():
        documents.append(
            Document(
                page_content=row_to_text(row),
                metadata={
                    "source_type": "excel",
                    "source_name": source_name,
                    "sheet_name": sheet_name,
                    "row_number": int(row["excel_row"]),
                    "incident_id": row["incident_id"],
                    "sap_module": row["sap_module"],
                    "priority": row["priority"],
                    "category": row["category"],
                    "owner_team": row["owner_team"],
                },
            )
        )
    return documents

## Task 4 — Row-aware chunking

Fixed-size chunking would wreck this data. A 500-character window over the sheet
would cut between `Root Cause:` and `Resolution:`, or merge the tail of INC-1004
with the head of INC-1005. Two concrete failures follow:

- **Severed fields** — a chunk with a root cause but no resolution retrieves well
  and answers nothing.
- **Blended records** — a chunk spanning two incidents lets the model attribute
  INC-1005's fix to INC-1004. That is a hallucination handed to it by retrieval.

So a record is split **only** if it exceeds ~1,500 characters. The longest row here
is 587 characters, so all 20 stay intact — the splitter is insurance for one
unusually verbose incident, not the normal path.

In [ ]:
def chunk_documents(
    documents: List[Document], max_chars: int = MAX_CHARS_PER_CHUNK
) -> List[Document]:
    """Keep one incident as one chunk unless its text is genuinely too long.

    Fixed-size chunking would cut a record between "Root Cause" and "Resolution"
    and hand the retriever half an incident. Splitting only oversized records
    keeps each chunk a self-contained, citable unit.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chars,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n", ". ", " ", ""],
    )

    chunks: List[Document] = []
    split_count = 0
    for document in documents:
        incident_id = document.metadata["incident_id"]
        if len(document.page_content) <= max_chars:
            parts = [document]
        else:
            parts = splitter.split_documents([document])
            split_count += 1

        for index, part in enumerate(parts, start=1):
            part.metadata = {
                **document.metadata,
                "chunk_id": f"{incident_id}-chunk-{index}",
                "chunk_index": index,
                "chunk_total": len(parts),
            }
            chunks.append(part)

    print("\n" + "=" * 70)
    print("TASKS 3-4 -- DOCUMENTS AND CHUNKS")
    print("=" * 70)
    print(f"\nDocuments: {len(documents)}   Chunks: {len(chunks)}")
    print(f"Records that needed splitting: {split_count}")
    print(f"Longest record: {max(len(d.page_content) for d in documents)} chars")
    return chunks

## Tasks 5 & 6 — Embeddings and the vector database

`gemini-embedding-001` turns each incident into a 3,072-dimension vector, and FAISS
indexes them alongside the text and metadata.

The `embed_query("HANA database performance issue")` probe is the Task 5 sanity
check: it proves a question and a record land in the same vector space.

In [ ]:
def build_vector_store(chunks: List[Document]) -> FAISS:
    """Embed the incident chunks with Gemini and index them in FAISS."""
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL)

    probe = embeddings.embed_query("HANA database performance issue")
    print("\n" + "=" * 70)
    print("TASKS 5-6 -- EMBEDDINGS AND VECTOR STORE")
    print("=" * 70)
    print(f"\nEmbedding model: {EMBED_MODEL}")
    print(f"Test question vector length: {len(probe)}")
    print(f"First five dimensions: {[round(v, 4) for v in probe[:5]]}")

    store = FAISS.from_documents(chunks, embeddings)
    print(f"Indexed {store.index.ntotal} vectors in FAISS.")
    return store

## Tasks 7 & 8 — Semantic retrieval and metadata filtering

**Task 7** is vector search: it matches *meaning*, not characters.

**Task 8** is the filter, and the two do different jobs. Similarity is a soft
signal, so "only P1" is not enforceable by embeddings — a P2 incident that *reads*
like a crisis can out-rank a terse P1. Priority is a recorded fact, so filtering on
metadata makes it a hard constraint and leaves similarity to do what it is good at:
ranking the survivors.

FAISS applies filters *after* the search, so `fetch_k` widens the candidate pool
first.

Scores are **L2 distances — lower is closer**, the opposite of a similarity score.

In [ ]:
@dataclass
class RetrievedIncident:
    """One retrieval hit, flattened for display and prompting."""

    rank: int
    score: float
    text: str
    metadata: dict

    @property
    def incident_id(self) -> str:
        return self.metadata["incident_id"]

    @property
    def row_number(self) -> int:
        return self.metadata["row_number"]


def retrieve(
    store: FAISS,
    question: str,
    top_k: int = 5,
    sap_module: Optional[str] = None,
    priority: Optional[str] = None,
    owner_team: Optional[str] = None,
) -> List[RetrievedIncident]:
    """Semantic search, optionally narrowed by structured metadata.

    Filters are applied inside the vector store, so "P1 SAP BTP incidents" can
    never return a P2 record that merely reads like one.
    """
    filters: Dict[str, str] = {}
    if sap_module:
        filters["sap_module"] = _canonical_module(sap_module)
    if priority:
        filters["priority"] = priority.strip().upper()
    if owner_team:
        filters["owner_team"] = owner_team.strip()

    # FAISS filters after the search, so fetch a wider candidate pool first.
    hits = store.similarity_search_with_score(
        question,
        k=top_k,
        filter=filters or None,
        fetch_k=max(50, top_k * 10),
    )
    return [
        RetrievedIncident(rank=i, score=float(score), text=doc.page_content, metadata=doc.metadata)
        for i, (doc, score) in enumerate(hits, start=1)
    ]


def print_retrieval(question: str, results: List[RetrievedIncident], show_text: bool = True) -> None:
    """Display Task 7's required fields for every hit."""
    print(f"\nQuery: {question}")
    if not results:
        print("  No incidents matched.")
        return
    for hit in results:
        print(f"\n  Rank {hit.rank} | score (L2 distance, lower is closer): {hit.score:.4f}")
        print(f"  Incident: {hit.incident_id} | Module: {hit.metadata['sap_module']} "
              f"| Priority: {hit.metadata['priority']} | Excel row: {hit.row_number}")
        print(f"  Chunk ID: {hit.metadata['chunk_id']}")
        if show_text:
            for line in hit.text.splitlines():
                print(f"      {line}")

## Task 9 — The grounded prompt

Four anti-hallucination mechanisms stack here:

1. **Closed-book instruction** — use only the retrieved records; never invent an
   incident, cause, team or number.
2. **A mandatory fallback** — one fixed sentence when the context falls short, so
   refusal is an easy explicit path rather than a failure the model improvises
   around.
3. **Forced citation** — every claim carries an incident ID and Excel row. A
   fabricated fact has no row to cite, and any slip is auditable by the reader.
4. **`temperature=0`** — no sampling creativity in what is really a lookup task.

In [ ]:
FALLBACK = (
    "I could not find sufficient information in the incident dataset to "
    "answer this question."
)

RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are the SAP Incident Knowledge Assistant. You answer support "
            "engineers using ONLY the historical incident records supplied in "
            "the context.\n\n"
            "Rules:\n"
            "1. Use only the retrieved records. Never invent an incident, a "
            "root cause, a resolution, a team or a number.\n"
            "2. Name the incident ID for every fact you state.\n"
            "3. Mention the SAP module and the priority of each incident you use.\n"
            "4. Explain the resolution steps that were actually recorded.\n"
            "5. Cite the source as: Source: {source_name}, Excel row <number>.\n"
            "6. If the context does not contain the answer, reply exactly with:\n"
            "   {fallback}\n"
            "   Do not add anything else in that case.\n"
            "7. If the question asks for something the incident data does not "
            "track at all, use the same fallback sentence.\n"
            "Answer in plain prose. Be concise and specific.",
        ),
        (
            "human",
            "Retrieved incident records:\n"
            "---------------------------\n"
            "{context}\n"
            "---------------------------\n\n"
            "Question: {question}",
        ),
    ]
)


def format_context(results: List[RetrievedIncident]) -> str:
    """Render retrieved records, each tagged with its citation coordinates."""
    blocks = []
    for hit in results:
        blocks.append(
            f"[Record {hit.rank}] "
            f"(source: {hit.metadata['source_name']}, sheet: {hit.metadata['sheet_name']}, "
            f"Excel row: {hit.row_number}, chunk: {hit.metadata['chunk_id']})\n"
            f"{hit.text}"
        )
    return "\n\n".join(blocks)

## Task 10 — The final RAG function

`ask_incident_rag(question, store, top_k=5, sap_module=None, priority=None)`
retrieves, formats the context, prompts Gemini, and returns the answer together
with the records it was built from.

`_response_text` handles both shapes of `response.content`: a plain string on most
replies, a list of content blocks on others.

In [ ]:
@dataclass
class RagAnswer:
    """A grounded answer plus the records it was built from."""

    question: str
    answer: str
    sources: List[RetrievedIncident]
    route: str = "semantic"

    def display(self) -> None:
        print(f"\nQ: {self.question}")
        print(f"   [route: {self.route}]")
        print(f"\nA: {self.answer}")
        if self.sources:
            print("\n   Sources used:")
            for hit in self.sources:
                print(
                    f"     - {hit.incident_id} | {hit.metadata['sap_module']} "
                    f"| {hit.metadata['priority']} | {hit.metadata['source_name']} "
                    f"row {hit.row_number}"
                )
        print("-" * 70)


def _chat_model() -> ChatGoogleGenerativeAI:
    # temperature=0 keeps the answer pinned to the retrieved text.
    return ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0)


def _response_text(response) -> str:
    """Flatten a chat response to plain text.

    ``content`` is a plain string on most replies but a list of content blocks
    on others, so both shapes have to be handled.
    """
    content = response.content
    if isinstance(content, str):
        return content.strip()
    parts = []
    for block in content:
        if isinstance(block, str):
            parts.append(block)
        elif isinstance(block, dict) and block.get("type") == "text":
            parts.append(block.get("text", ""))
    return "".join(parts).strip()


def ask_incident_rag(
    question: str,
    store: FAISS,
    top_k: int = 5,
    sap_module: Optional[str] = None,
    priority: Optional[str] = None,
    owner_team: Optional[str] = None,
) -> RagAnswer:
    """Answer a question from the incident history, grounded in retrieved rows."""
    results = retrieve(
        store,
        question,
        top_k=top_k,
        sap_module=sap_module,
        priority=priority,
        owner_team=owner_team,
    )

    if not results:
        return RagAnswer(question=question, answer=FALLBACK, sources=[])

    chain = RAG_PROMPT | _chat_model()
    response = chain.invoke(
        {
            "context": format_context(results),
            "question": question,
            "fallback": FALLBACK,
            "source_name": results[0].metadata["source_name"],
        }
    )
    return RagAnswer(question=question, answer=_response_text(response), sources=results)

## Task 11 — Routing analytical questions to Pandas

**This is the limitation the problem statement asks you to address.**

*"How many P1 incidents were reported?"* is unanswerable by retrieval, by
construction. Top-k returns 5 rows; the answer is 6. The model would faithfully
count what it was given and confidently report **5** — grounded in its context and
still wrong. Averages fail worse: an average over the 5 most *similar* rows is an
average over a biased sample nobody asked for.

The fix is not a better prompt, it is a different engine. `compute_statistics` runs
Pandas over the **whole** dataframe and Gemini only phrases the exact numbers.
Both branches keep the same fallback sentence, so refusal behaviour holds either
way.

In [ ]:
class Route(BaseModel):
    """Which engine should answer this question."""

    route: str = Field(
        description=(
            "'analytical' when the question needs a calculation over ALL "
            "incidents -- counts, averages, totals, min/max across the dataset, "
            "or ranking every module. 'semantic' when the question is about the "
            "content of specific incidents, their causes or resolutions."
        )
    )
    reason: str = Field(description="One short sentence explaining the choice.")


ANALYTICAL_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer questions about an SAP incident dataset using ONLY the "
            "pre-computed statistics below. They were calculated with Pandas over "
            "every row, so they are exact -- quote the numbers as given and do "
            "not recompute or estimate. If the statistics do not contain the "
            "answer, reply exactly with:\n{fallback}\n"
            "Cite the dataset as: Source: {source_name} (computed with Pandas "
            "over all rows). Be concise.",
        ),
        (
            "human",
            "Dataset statistics:\n"
            "-------------------\n"
            "{stats}\n"
            "-------------------\n\n"
            "Question: {question}",
        ),
    ]
)


def classify_question(question: str) -> Route:
    """Decide whether vector RAG or Pandas should handle the question."""
    model = _chat_model().with_structured_output(Route)
    return model.invoke(
        "Classify this question about an SAP incident dataset as 'analytical' "
        f"or 'semantic'.\n\nQuestion: {question}"
    )


def compute_statistics(df: pd.DataFrame) -> str:
    """Exact aggregates from Pandas -- the numbers vector search cannot give.

    Retrieval returns the top-k most similar rows, never "all of them", so any
    count or average built from retrieved context is wrong by construction. These
    are computed over the full dataframe instead.
    """
    lines = [f"Total incidents: {len(df)}", ""]

    lines.append("Incident count by priority:")
    for priority, count in df["priority"].value_counts().sort_index().items():
        lines.append(f"  {priority}: {count}")

    lines.append("\nIncident count by SAP module:")
    for module, count in df["sap_module"].value_counts().items():
        lines.append(f"  {module}: {count}")

    lines.append("\nResolution time (hours) by SAP module:")
    stats = df.groupby("sap_module")["resolution_time_hours"].agg(["mean", "max", "sum", "count"])
    for module, row in stats.iterrows():
        lines.append(
            f"  {module}: average {row['mean']:.2f}, longest {row['max']:.2f}, "
            f"total {row['sum']:.2f}, incidents {int(row['count'])}"
        )

    lines.append("\nResolution time (hours) by priority:")
    stats = df.groupby("priority")["resolution_time_hours"].agg(["mean", "max", "count"])
    for priority, row in stats.sort_index().iterrows():
        lines.append(
            f"  {priority}: average {row['mean']:.2f}, longest {row['max']:.2f}, "
            f"incidents {int(row['count'])}"
        )

    slowest = df.loc[df["resolution_time_hours"].idxmax()]
    lines.append(
        f"\nLongest incident overall: {slowest['incident_id']} "
        f"({slowest['sap_module']}, {slowest['priority']}, "
        f"{slowest['resolution_time_hours']:g} hours, Excel row {slowest['excel_row']})"
    )

    lines.append("\nEvery incident (id | module | priority | hours | Excel row):")
    for _, row in df.iterrows():
        hours = row["resolution_time_hours"]
        hours_text = "unknown" if pd.isna(hours) else f"{hours:g}"
        lines.append(
            f"  {row['incident_id']} | {row['sap_module']} | {row['priority']} "
            f"| {hours_text} | row {row['excel_row']}"
        )

    return "\n".join(lines)


def ask_analytical(question: str, df: pd.DataFrame) -> RagAnswer:
    """Answer a calculation question from Pandas aggregates."""
    chain = ANALYTICAL_PROMPT | _chat_model()
    response = chain.invoke(
        {
            "stats": compute_statistics(df),
            "question": question,
            "fallback": FALLBACK,
            "source_name": df.attrs.get("source_name", EXCEL_FILE),
        }
    )
    return RagAnswer(
        question=question,
        answer=_response_text(response),
        sources=[],
        route="analytical (Pandas)",
    )


def answer_question(
    question: str,
    store: FAISS,
    df: pd.DataFrame,
    top_k: int = 5,
    **filters,
) -> RagAnswer:
    """Front door: classify the question, then send it to RAG or to Pandas."""
    if filters.get("sap_module") or filters.get("priority") or filters.get("owner_team"):
        # An explicit filter means the caller already wants record lookup.
        return ask_incident_rag(question, store, top_k=top_k, **filters)

    route = classify_question(question)
    if route.route == "analytical":
        return ask_analytical(question, df)
    return ask_incident_rag(question, store, top_k=top_k, **filters)

## Assembly

`build_assistant` runs Tasks 1–6 and hands back the vector store plus the clean
dataframe — the store for semantic questions, the dataframe for analytical ones.

In [ ]:
def build_assistant(path: str = EXCEL_FILE, verbose: bool = True):
    """Run the whole pipeline and return the vector store plus clean dataframe."""
    raw = load_incidents(path)
    if verbose:
        explore_incidents(raw)

    clean = clean_incidents(raw)
    clean.attrs.update(raw.attrs)

    documents = rows_to_documents(clean)
    chunks = chunk_documents(documents)

    if verbose:
        print("\nExample document (INC-1006):")
        for doc in documents:
            if doc.metadata["incident_id"] == "INC-1006":
                print(doc.page_content)
                print(f"\nMetadata: {doc.metadata}")
                break

    store = build_vector_store(chunks)
    return store, clean

## Run the pipeline

Tasks 1–6 end to end: load, explore, clean, convert to documents, chunk, embed and
index.

In [ ]:
store, df = build_assistant()

## Task 7 — Semantic retrieval

Note the wording of these questions: it deliberately differs from the text in the
sheet. *"HANA memory exhaustion"* is not a phrase any row contains, yet INC-1002
ranks first — because the embedding matches meaning, not characters. A keyword
search for that phrase returns nothing.

In [ ]:
for question in [
    "Which incident was related to HANA memory exhaustion?",
    "Find incidents involving SAP BTP connectivity problems.",
    "Which issue was caused by an incorrect pricing procedure?",
    "Show incidents related to employee integration failures.",
]:
    print_retrieval(question, retrieve(store, question, top_k=3))

## Task 8 — Metadata filtering

The filter is a hard constraint: a non-P1 row is now structurally unreachable, no
matter how similar it reads.

In [ ]:
for question, filters in [
    ("Show the most critical incidents.", {"priority": "P1"}),
    ("Database problems.", {"sap_module": "SAP HANA"}),
    ("Access and connection issues.",
     {"sap_module": "SAP BTP", "owner_team": "BTP Platform Team"}),
]:
    print(f"\nFilters: {filters}")
    print_retrieval(question, retrieve(store, question, top_k=3, **filters),
                    show_text=False)

## Mandatory test questions

Every question from section 7 of the problem statement — factual, semantic,
comparison, filter-based, recommendation-style, and one the data cannot answer.

`answer_question` classifies each one first, so watch the `[route: ...]` line:
*"Which P1 incident took the longest time to resolve?"* is a calculation and goes
to Pandas, while the rest go to vector RAG.

In [ ]:
MANDATORY_QUESTIONS = [
    # Direct factual
    ("What was the resolution for incident INC-1004?", {}),
    ("Which team resolved the employee replication issue?", {}),
    # Semantic
    ("Has there been an issue where a cloud application could not connect to an SAP backend?", {}),
    ("Find a previous incident involving database memory problems.", {}),
    # Comparison
    ("Which P1 incident took the longest time to resolve?", {}),
    ("Compare the two SAP HANA P1 incidents.", {}),
    # Filter-based
    ("Show only P1 SAP BTP incidents.", {"sap_module": "SAP BTP", "priority": "P1"}),
    ("Find SAP MM incidents handled by the Procure-to-Pay Support team.",
     {"sap_module": "SAP MM", "owner_team": "Procure-to-Pay Support"}),
    # Recommendation-style
    ("A user reports that a BTP application cannot access the backend because "
     "authentication is failing. Which previous incident is most similar, and "
     "what resolution should the support team investigate?", {}),
    # Unsupported -- must refuse rather than invent
    ("What is the annual revenue of the company?", {}),
]

for question, filters in MANDATORY_QUESTIONS:
    answer_question(question, store, df, **filters).display()

### The unsupported question

The dataset tracks incidents, not finances, so *"What is the annual revenue of the
company?"* returns the fallback sentence rather than a guess. That refusal is the
grounded prompt working.

## Section 11 — Analytical questions

These are the questions vector retrieval alone gets *confidently wrong*. Each is
routed to Pandas and computed over every row.

You can verify the numbers yourself in the cell below.

In [ ]:
for question in [
    "What is the average resolution time for SAP HANA incidents?",
    "How many P1 incidents were reported?",
    "Which module has the highest average resolution time?",
]:
    answer_question(question, store, df).display()

In [ ]:
# Check the LLM's numbers against Pandas directly.
print("Average resolution time by module:")
print(df.groupby("sap_module")["resolution_time_hours"].mean().round(2).to_string())
print("\nP1 incidents:", (df["priority"] == "P1").sum())

## Section 8 — Expected final output

The worked example from the problem statement. Expected: INC-1006, 9 hours,
`sap_incidents.xlsx` Excel row 7.

In [ ]:
ask_incident_rag(
    "Which P1 SAP HANA incident took the longest to resolve?",
    store,
    sap_module="SAP HANA",
    priority="P1",
).display()

## Limitations and future improvements

**Aggregation at scale.** The Pandas branch works here because 20 incidents
summarise into the prompt. At thousands of rows that digest stops fitting in the
context window, and the aggregate step should become a real tool call — a
`pandas.query` tool or SQL — with the LLM writing the query instead of reading a
pre-computed digest.

**Other known limits**

- Retrieval has no score threshold: every query returns `top_k` rows even when
  nothing is relevant. The grounded prompt catches this at the answer stage, but a
  distance floor would stop weak context being sent at all.
- The classifier is one LLM call with no fallback; a misrouted count answers from 5
  rows without saying so.
- FAISS filters post-search, so on a much larger sheet a narrow filter could exhaust
  the candidate pool. Chroma's native `where` clause would scale better.
- Filters are exact-match only — no ranges (`resolution_time_hours > 5`), no OR.
- No conversation memory: *"and who fixed that one?"* will not resolve.
- The index rebuilds on every run; `FAISS.save_local` would persist it.

**Worth adding next:** hybrid keyword + vector search (an exact ID like `INC-1004`
is really a keyword problem), reranking, query rewriting, conversation memory, a
Streamlit UI, SAP HANA Cloud Vector Engine in place of FAISS, deployment to SAP BTP
Cloud Foundry, and an evaluation set with automatic groundedness checks.